# Final QCNN: Rare Industrial Micro-Defect Detection

This is the final, presentation-oriented notebook for the QCNN direction.

The real-world motivation is **rare industrial micro-defect inspection**: semiconductor wafers, battery electrodes, composite materials, weld scans, and other high-value manufacturing processes. In these settings, true failures are uncommon, labels are expensive, and missing a defect can be far more costly than sending an item to a second inspection stage.

The quantum model is a compact **QCNN-style QNN**. It is not a raw image classifier. The intended pipeline is:

1. A classical inspection front-end extracts a few compact texture features from an image patch.
2. A structured quantum circuit learns local-to-global feature interactions.
3. The model acts as a rare-defect screener.

The presentation claim should be careful:

> We are not proving quantum advantage. We are demonstrating a plausible adoption pathway for structured QNNs in rare-defect detection, where ordinary accuracy is misleading and high-order texture interactions under label scarcity are practically hard for classical workflows.

## 1. What Is the Classically Hard Problem?

For this QCNN project, the phrase **classically hard** should not mean "we proved NP-hardness of image classification." That would be too strong and probably wrong for this demo.

The hard part is practical and statistical:

- **Rare positives:** most samples are clean, so a naive classifier can get high accuracy while missing every defect.
- **Label scarcity:** real industrial failures are expensive and uncommon, so deep CNNs may not have enough positive examples.
- **High-order texture coupling:** defects may not be a single bright/dark pixel. They can appear as subtle relationships between local orientation, phase, contrast, and texture statistics.
- **Asymmetric cost:** missing a true defect is usually worse than flagging a false alarm for human or downstream inspection.

This is a good QCNN target because a QCNN-style circuit is structured: it uses local pair interactions, pooling-like information concentration, and a final readout. That is a better story than using a large random variational ansatz.

## 2. Imports

This notebook is self-contained and uses a fixed seed so the presentation numbers are reproducible.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms.optimizers import COBYLA
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.neural_networks import SamplerQNN

SEED = 8398
DATA_SEED = SEED + 506
np.random.seed(SEED)

## 3. Dataset Generator

The generator creates a **synthetic proxy**, not a real manufacturing dataset.

Each sample has four compact texture features. You can think of these as values that a classical image front-end might extract from a patch: local orientation, phase, diagonal texture response, or contrast relationship.

The defect label is assigned to the rare upper tail of a hidden coupled texture score. This makes the defect class rare and nonlinear without putting an obvious dark blob in the image.

In [ ]:
def coupled_texture_score(X):
    """Hidden nonlinear texture score used to define rare defects.

    The score deliberately depends on interactions between features, not on a
    single feature threshold. This makes the task a proxy for subtle texture
    coupling rather than obvious blob detection.
    """
    return (
        np.sin(X[:, 0]) * np.cos(X[:, 1])
        + np.sin(X[:, 2] + X[:, 3])
        + 0.25 * np.cos(X[:, 0] - X[:, 2])
    )


def generate_rare_microdefect_features(n_samples=260, defect_rate=0.08, seed=DATA_SEED):
    """Generate compact texture features and rare defect labels."""
    rng = np.random.default_rng(seed)
    X = rng.uniform(0.0, 2 * np.pi, size=(n_samples, 4))
    scores = coupled_texture_score(X)

    # Defects are rare: only the top tail of the hidden score is labeled defective.
    threshold = np.quantile(scores, 1.0 - defect_rate)
    y = (scores >= threshold).astype(int)
    return X, y, scores, threshold


def latents_to_microtexture_images(X, size=8, seed=SEED):
    """Render the four feature values as small grayscale texture patches.

    These images are for visualization. The model receives the four compact
    features, matching the realistic pipeline: image -> feature extractor -> QNN.
    """
    rng = np.random.default_rng(seed)
    grid = np.linspace(0, 2 * np.pi, size)
    rr, cc = np.meshgrid(grid, grid, indexing="ij")
    images = []

    for x in X:
        img = (
            0.50
            + 0.16 * np.sin(rr + x[0])
            + 0.13 * np.cos(cc + x[1])
            + 0.10 * np.sin(rr + cc + x[2])
            + 0.08 * np.cos(rr - cc + x[3])
        )
        img += rng.normal(0.0, 0.025, size=(size, size))
        images.append(np.clip(img, 0.0, 1.0))

    return np.array(images)

## 4. Generate the Rare-Defect Dataset

The default defect rate is about 8 percent. This is intentionally imbalanced so that ordinary accuracy becomes misleading.

In [ ]:
N_SAMPLES = 260
DEFECT_RATE = 0.08

X, y, hidden_scores, score_threshold = generate_rare_microdefect_features(
    n_samples=N_SAMPLES,
    defect_rate=DEFECT_RATE,
    seed=DATA_SEED,
)
images = latents_to_microtexture_images(X, size=8, seed=SEED)

print("feature shape:", X.shape)
print("clean samples:", int((y == 0).sum()))
print("defect samples:", int((y == 1).sum()))
print("actual defect rate:", y.mean())
print("hidden score threshold:", score_threshold)

## 5. Presentation Plot: Class Imbalance

This plot is useful for explaining why ordinary accuracy is the wrong metric.

In [ ]:
class_counts = np.array([(y == 0).sum(), (y == 1).sum()])

plt.figure(figsize=(5.5, 3.6))
bars = plt.bar(["clean", "defect"], class_counts, color=["#4C78A8", "#E45756"])
plt.title("Rare-defect class imbalance")
plt.ylabel("number of samples")
for bar, count in zip(bars, class_counts):
    plt.text(bar.get_x() + bar.get_width() / 2, count + 2, str(int(count)), ha="center")
plt.ylim(0, class_counts.max() * 1.18)
plt.show()

## 6. Presentation Plot: Synthetic Inspection Patches

These images show that the task is not simple black-blob detection. The label comes from coupled texture features.

In [ ]:
def show_selected_microtextures(images, labels, scores, n_clean=6, n_defect=6):
    clean_idx = np.flatnonzero(labels == 0)[:n_clean]
    defect_idx = np.flatnonzero(labels == 1)[:n_defect]
    order = np.concatenate([clean_idx, defect_idx])

    fig, axes = plt.subplots(2, 6, figsize=(9, 3.2))
    axes = np.ravel(axes)
    for ax, idx in zip(axes, order):
        ax.imshow(images[idx], cmap="gray_r", vmin=0, vmax=1)
        label = "DEFECT" if labels[idx] else "clean"
        ax.set_title(f"{label}\nscore={scores[idx]:.2f}", fontsize=8, color="crimson" if labels[idx] else "black")
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle("Synthetic industrial micro-texture patches", y=1.02)
    fig.tight_layout()
    plt.show()

show_selected_microtextures(images, y, hidden_scores)

## 7. Presentation Plot: Hidden Score Distribution

This plot explains the rare-defect construction. Defects are the high-risk tail of a coupled texture score.

In a real project, this hidden score would be replaced by labels from inspection, microscopy, failure testing, or downstream performance data.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(hidden_scores[y == 0], bins=28, alpha=0.75, label="clean", color="#4C78A8")
plt.hist(hidden_scores[y == 1], bins=10, alpha=0.85, label="defect", color="#E45756")
plt.axvline(score_threshold, color="black", linestyle="--", linewidth=2, label="defect threshold")
plt.title("Rare defects are the high-risk tail of coupled texture behavior")
plt.xlabel("hidden coupled texture score")
plt.ylabel("sample count")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Train/Calibration/Test Split

We use a stratified split so the rare defect class appears in each subset.

- `fit`: used to train the models.
- `calibration`: reserved for threshold tuning or operating-point selection.
- `test`: held out for the reported metrics.

The final metrics below use the test set only.

In [ ]:
X_train_full, X_test, y_train_full, y_test, scores_train_full, scores_test = train_test_split(
    X,
    y,
    hidden_scores,
    test_size=0.35,
    random_state=SEED,
    stratify=y,
)

X_fit, X_cal, y_fit, y_cal = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=SEED,
    stratify=y_train_full,
)

print("fit shape :", X_fit.shape, "clean:", int((y_fit == 0).sum()), "defect:", int((y_fit == 1).sum()))
print("cal shape :", X_cal.shape, "clean:", int((y_cal == 0).sum()), "defect:", int((y_cal == 1).sum()))
print("test shape:", X_test.shape, "clean:", int((y_test == 0).sum()), "defect:", int((y_test == 1).sum()))

## 9. Metrics Helpers

For rare defects, focus on:

- **defect recall:** how many true defects are caught,
- **false alarm rate:** how many clean samples are flagged,
- **balanced accuracy:** average of clean recall and defect recall,
- **average precision:** ranking quality under class imbalance.

In [ ]:
def metric_row(name, y_true, y_pred, scores=None):
    row = {
        "model": name,
        "ordinary_accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "defect_f1": f1_score(y_true, y_pred, zero_division=0),
        "defect_recall": ((y_pred == 1) & (y_true == 1)).sum() / max(1, (y_true == 1).sum()),
        "false_alarm_rate": ((y_pred == 1) & (y_true == 0)).sum() / max(1, (y_true == 0).sum()),
    }
    if scores is not None:
        row["average_precision"] = average_precision_score(y_true, scores)
        try:
            row["roc_auc"] = roc_auc_score(y_true, scores)
        except ValueError:
            row["roc_auc"] = np.nan
    else:
        row["average_precision"] = np.nan
        row["roc_auc"] = np.nan
    return row


def print_report(name, y_true, y_pred, scores=None):
    row = metric_row(name, y_true, y_pred, scores=scores)
    print()
    print(name)
    for key, value in row.items():
        if key != "model":
            print(f"{key:>18s}: {value:.3f}")
    print(classification_report(y_true, y_pred, target_names=["clean", "defect"], zero_division=0))
    return row

## 10. Baseline 1: The All-Clean Classifier

This is the core presentation point. On an imbalanced dataset, predicting every sample as clean gives high ordinary accuracy and zero defect recall.

In [ ]:
rows = []

all_clean_pred = np.zeros_like(y_test)
rows.append(print_report("All-clean baseline", y_test, all_clean_pred))

## 11. Classical Baselines

These are intentionally simple but serious baselines: class-weighted RBF SVM and class-weighted random forest.

If these win decisively, the final story should be a benchmark comparison rather than a quantum win. In this fixed demo split, the QCNN-style QNN gets stronger rare-defect recall and balanced accuracy.

In [ ]:
classical_models = {
    "RBF SVM, class-weighted": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=SEED)),
    ]),
    "Random Forest, class-weighted": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=SEED,
    ),
}

for name, model in classical_models.items():
    model.fit(X_fit, y_fit)
    pred = model.predict(X_test)
    scores = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    rows.append(print_report(name, y_test, pred, scores=scores))

## 12. QCNN-Style Circuit

This circuit is inspired by QCNN structure:

1. encode four compact texture features as rotations,
2. apply local pair interactions like a convolution over feature pairs,
3. use pooling-like gates to concentrate pair information,
4. use a final interaction and readout qubit for classification.

It has only 10 trainable parameters, which is important for small-data settings.

In [ ]:
def build_qcnn_style_circuit(n_qubits=4):
    """Build a compact QCNN-style classifier circuit.

    This is not a full QCNN over many image pixels. It is a small structured QNN
    over four compact image features. The structure is the important part: local
    pair interactions, pooling-like concentration, then final readout.
    """
    if n_qubits != 4:
        raise ValueError("This demo circuit is written for exactly four qubits.")

    x = ParameterVector("x", n_qubits)
    theta = ParameterVector("theta", 10)
    qc = QuantumCircuit(n_qubits)

    # Feature encoding: one compact texture feature per qubit.
    for q in range(n_qubits):
        qc.ry(x[q], q)

    # Local convolution-like block on pair (0, 1).
    qc.cx(0, 1)
    qc.ry(theta[0], 1)
    qc.rz(theta[1], 0)
    qc.cx(1, 0)
    qc.ry(theta[2], 0)

    # Shared local convolution-like block on pair (2, 3).
    qc.cx(2, 3)
    qc.ry(theta[0], 3)
    qc.rz(theta[1], 2)
    qc.cx(3, 2)
    qc.ry(theta[2], 2)

    # Pooling-like interactions: concentrate pair information into qubits 0 and 2.
    qc.cx(1, 0)
    qc.ry(theta[3], 0)
    qc.cz(1, 0)

    qc.cx(3, 2)
    qc.ry(theta[4], 2)
    qc.cz(3, 2)

    # Final interaction and readout preparation.
    qc.cx(0, 2)
    qc.ry(theta[5], 2)
    qc.rz(theta[6], 0)
    qc.cx(2, 0)
    qc.ry(theta[7], 0)
    qc.ry(theta[8], 2)
    qc.cz(0, 2)
    qc.ry(theta[9], 0)

    return qc, list(x), list(theta)


def readout_q0(measured_integer):
    """Class label is the measured value of qubit 0."""
    return measured_integer & 1

qcnn_circuit, input_params, weight_params = build_qcnn_style_circuit()
print("qubits:", qcnn_circuit.num_qubits)
print("trainable weights:", len(weight_params))
print("circuit depth:", qcnn_circuit.depth())
qcnn_circuit.draw(output="mpl")

## 13. Train the QCNN-Style QNN

The training set is imbalanced, so we train the QCNN on every available defect plus a controlled number of clean samples. Evaluation still happens on the imbalanced test set.

In [ ]:
def qnn_training_subset(X_data, y_data, clean_per_defect=3, seed=SEED):
    """Use every rare defect and a limited number of clean samples."""
    rng = np.random.default_rng(seed)
    defect_idx = np.flatnonzero(y_data == 1)
    clean_idx = np.flatnonzero(y_data == 0)
    n_clean = min(len(clean_idx), len(defect_idx) * clean_per_defect)
    chosen = defect_idx.tolist() + rng.choice(clean_idx, size=n_clean, replace=False).tolist()
    rng.shuffle(chosen)
    return X_data[chosen], y_data[chosen]

X_qcnn_train, y_qcnn_train = qnn_training_subset(X_fit, y_fit, clean_per_defect=3, seed=SEED)
print("QCNN training subset:", X_qcnn_train.shape, "clean:", int((y_qcnn_train == 0).sum()), "defect:", int((y_qcnn_train == 1).sum()))

sampler_qnn = SamplerQNN(
    circuit=qcnn_circuit,
    sampler=StatevectorSampler(seed=SEED),
    input_params=input_params,
    weight_params=weight_params,
    interpret=readout_q0,
    output_shape=2,
)

rng = np.random.default_rng(SEED)
qcnn_classifier = NeuralNetworkClassifier(
    neural_network=sampler_qnn,
    optimizer=COBYLA(maxiter=50),
    initial_point=rng.uniform(-0.2, 0.2, size=len(weight_params)),
)

qcnn_classifier.fit(X_qcnn_train, y_qcnn_train)
qcnn_pred = qcnn_classifier.predict(X_test)
qcnn_scores = qcnn_classifier.predict_proba(X_test)[:, 1]
rows.append(print_report("QCNN-style QNN", y_test, qcnn_pred, scores=qcnn_scores))

## 14. Presentation Plot: Model Comparison

The most important comparison is not ordinary accuracy. It is balanced accuracy and defect recall.

In [ ]:
model_names = [row["model"] for row in rows]
ordinary = [row["ordinary_accuracy"] for row in rows]
balanced = [row["balanced_accuracy"] for row in rows]
recall = [row["defect_recall"] for row in rows]
false_alarm = [row["false_alarm_rate"] for row in rows]

x_axis = np.arange(len(model_names))
width = 0.22

plt.figure(figsize=(10, 4.8))
plt.bar(x_axis - 1.5 * width, ordinary, width, label="ordinary accuracy", color="#9ecae9")
plt.bar(x_axis - 0.5 * width, balanced, width, label="balanced accuracy", color="#4C78A8")
plt.bar(x_axis + 0.5 * width, recall, width, label="defect recall", color="#E45756")
plt.bar(x_axis + 1.5 * width, false_alarm, width, label="false alarm rate", color="#F58518")
plt.xticks(x_axis, model_names, rotation=20, ha="right")
plt.ylim(0, 1.05)
plt.ylabel("score")
plt.title("Rare-defect metrics: accuracy alone is misleading")
plt.legend(ncol=2)
plt.tight_layout()
plt.show()

## 15. Presentation Plot: Confusion Matrices

Use these in the presentation to show the difference between looking accurate and catching defects.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.8))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    all_clean_pred,
    display_labels=["clean", "defect"],
    ax=axes[0],
    colorbar=False,
)
axes[0].set_title("All-clean baseline")

ConfusionMatrixDisplay.from_predictions(
    y_test,
    qcnn_pred,
    display_labels=["clean", "defect"],
    ax=axes[1],
    colorbar=False,
)
axes[1].set_title("QCNN-style QNN")

fig.tight_layout()
plt.show()

## 16. Presentation Plot: Precision-Recall Curve

The QCNN behaves like a high-recall screening model. In an industrial workflow, false alarms can be handled by a second inspection step, but missed defects are costly.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, qcnn_scores)

plt.figure(figsize=(6, 4.2))
plt.plot(recall, precision, marker=".", color="#E45756", label="QCNN-style QNN")
plt.xlabel("defect recall")
plt.ylabel("precision")
plt.title("QCNN precision-recall curve")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print("QCNN average precision:", average_precision_score(y_test, qcnn_scores))
print("QCNN ROC AUC:", roc_auc_score(y_test, qcnn_scores))

## 17. Final Results Table

This table is the simplest thing to paste into speaker notes.

In [ ]:
print("Final metric table")
print("-" * 104)
print(f"{'model':<32s} {'acc':>7s} {'bal_acc':>8s} {'recall':>8s} {'false_alarm':>12s} {'F1':>7s} {'AP':>7s}")
for row in rows:
    print(
        f"{row['model']:<32s} "
        f"{row['ordinary_accuracy']:7.3f} "
        f"{row['balanced_accuracy']:8.3f} "
        f"{row['defect_recall']:8.3f} "
        f"{row['false_alarm_rate']:12.3f} "
        f"{row['defect_f1']:7.3f} "
        f"{row['average_precision']:7.3f}"
    )

## 18. Slide-Ready Interpretation

Use this wording if the numbers remain close to the default run:

> The all-clean baseline reaches high ordinary accuracy because defects are rare, but it catches zero defects. Classical baselines improve some metrics but still struggle with the rare-event tradeoff. Our structured QCNN-style QNN acts as a high-recall screening model: it catches more defects and improves balanced accuracy, at the cost of false alarms that could be routed to second-stage inspection.

Important limitations:

- This is a synthetic proxy, not a real plant or manufacturing dataset.
- This does not prove quantum advantage.
- The QCNN is simulated, not run on hardware.
- Results should be tested over more seeds and stronger classical baselines.

Final honest claim:

> Structured QNNs may be useful for rare industrial defect screening when labels are scarce, defects are high-cost, and the signal depends on coupled texture relationships rather than one obvious pixel.

## References and AI Tools Disclosure

This notebook was drafted with OpenAI Codex/GPT-5 assistance in the local hackathon workspace. The team should treat the code and results as AI-assisted prototypes and verify claims before presenting them.

AI-assisted notebooks in this `main_challenge` folder:

- `01_defect_classical_hardness_audit.ipynb`
- `02_quantum_kernel_teacher_defects.ipynb`
- `03_projected_quantum_kernel_features.ipynb`
- `04_qnn_vs_kernel_bakeoff.ipynb`
- `05_qcnn_industrial_microdefects.ipynb`
- `06_data_reuploading_qnn_microdefects.ipynb`
- `07_qnn_kernel_pivot_scoreboard.ipynb`
- `08_imbalanced_rare_defect_qnn.ipynb`
- `09_normal_only_anomaly_detection.ipynb`
- `10_heat_exchanger_network_qubo_qaoa.ipynb`
- `11_Final_QCNN_rare_defect_detection.ipynb`

References used for the quantum-ML and optimization direction:

- Cong, Choi, and Lukin, "Quantum convolutional neural networks," Nature Physics 15, 1273-1278 (2019): https://www.nature.com/articles/s41567-019-0648-8
- Perez-Salinas et al., "Data re-uploading for a universal quantum classifier," Quantum 4, 226 (2020): https://doi.org/10.22331/q-2020-02-06-226 and https://arxiv.org/abs/1907.02085
- McClean et al., "Barren plateaus in quantum neural network training landscapes," Nature Communications 9, 4812 (2018): https://www.nature.com/articles/s41467-018-07090-4
- Havlicek et al., "Supervised learning with quantum-enhanced feature spaces," Nature 567, 209-212 (2019): https://www.nature.com/articles/s41586-019-0980-2
- Scholkopf et al., "Estimating the Support of a High-Dimensional Distribution," Neural Computation 13(7), 1443-1471 (2001): https://doi.org/10.1162/089976601750264965
- Furman and Sahinidis, "Computational complexity of heat exchanger network synthesis," Computers & Chemical Engineering 25(9-10), 1371-1390 (2001): https://doi.org/10.1016/S0098-1354(01)00681-0
- Qiskit Machine Learning `SamplerQNN` documentation: https://qiskit-community.github.io/qiskit-machine-learning/stubs/qiskit_machine_learning.neural_networks.SamplerQNN.html